# Supabase client and get data
This notebook demonstrates how to connect a local Ollama embedding model to a Supabase Postgres database with `pgvector` for semantic search over insurance-related text. It loads Supabase credentials from environment variables, initializes an `OllamaEmbeddings` instance (`mxbai-embed-large`), and defines a `search_supabase` helper that embeds a natural language query, calls the `match_documents` RPC function in Supabase, and prints the most similar stored documents with their similarity scores and metadata. An additional (commented) section shows how to ingest new texts by embedding them locally and inserting both the embeddings and metadata into the `documents` table via the Supabase Python client.

In [2]:
import os
from dotenv import load_dotenv
from supabase.client import create_client
from langchain_ollama import OllamaEmbeddings

load_dotenv()

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_SERVICE_ROLE_KEY"),
)

embeddings_model = OllamaEmbeddings(model="mxbai-embed-large")


def search_supabase(query: str, match_count: int = 3, filter: dict | None = None):
    if filter is None:
        filter = {}

    # 1) Embed the query with your local Ollama model
    query_embedding = embeddings_model.embed_query(query)

    # 2) Call your Postgres function via RPC
    response = supabase.rpc(
        "match_documents",
        {
            "query_embedding": query_embedding,
            "match_count": match_count,
            "filter": filter,
        },
    ).execute()

    return response.data

In [3]:
query = "How do I submit an insurance claim?"
results = search_supabase(query, match_count=3)

for r in results:
    print("content:", r["content"])
    print("similarity:", r["similarity"])
    print("metadata:", r["metadata"])
    print("-" * 40)

content: To submit an insurance claim, provide your policy number, invoice, and supporting medical documents.
similarity: 0.844581174887201
metadata: {'source': 'demo3'}
----------------------------------------
content: Claim processing usually requires policy number and invoice.
similarity: 0.715870997782818
metadata: {'source': 'demo2'}
----------------------------------------
content: Claim processing usually requires policy number and invoice.
similarity: 0.715870997782818
metadata: {'source': 'demo2'}
----------------------------------------


# Ingest data to Supabse

In [ ]:
# import os
# from dotenv import load_dotenv
# from supabase.client import create_client
# from langchain_ollama import OllamaEmbeddings

# load_dotenv()

# supabase = create_client(
#     os.getenv("SUPABASE_URL"),
#     os.getenv("SUPABASE_SERVICE_ROLE_KEY"),
# )

# embeddings_model = OllamaEmbeddings(model="mxbai-embed-large")


# def ingest_texts_to_supabase(texts, metadatas=None):
#     if metadatas is None:
#         metadatas = [{} for _ in texts]

#     rows = []
#     for text, metadata in zip(texts, metadatas):
#         embedding = embeddings_model.embed_query(text)
#         rows.append({
#             "content": text,
#             "metadata": metadata,
#             "embedding": embedding
#         })

#     response = supabase.table("documents").insert(rows).execute()
#     return response

In [ ]:
# response = ingest_texts_to_supabase(
#     texts=[
#         "Health insurance covers medically necessary treatments.",
#         "Claim processing usually requires policy number and invoice.",
#         "To submit an insurance claim, provide your policy number, invoice, and supporting medical documents."
#     ],
#     metadatas=[
#         {"source": "demo1"},
#         {"source": "demo2"},
#         {"source": "demo3"}
#     ]
# )

# print(response)